# 🤖 XAS AI Agent

Talk to this agent in natural language to visualize and analyze your XAS data.

**Example commands:**
- *"List all scans"*
- *"Show me the TEY for scan 45612"*
- *"Plot MCP of 45645 normalized by I0"*
- *"Compare TEY of scans 45611, 45612, 45613"*
- *"Save the last plot as my_data.txt"*
- *"Tell me about scan 45645"*

In [1]:
# ── Setup (no file loading — instant startup) ─────────────────────────────────
import os, sys, json, textwrap, importlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, os.path.abspath("."))
import xas_utils as xu
importlib.reload(xu)  # always pick up latest version

matplotlib.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
%matplotlib inline

# ── Load API key & configure LLM from .env ───────────────────────────────────
load_dotenv()

# Import the multi-provider LLM configuration from chat_app
import chat_app as _ca
importlib.reload(_ca)
client = _ca.client
MODEL = _ca.MODEL
DATA_DIR = _ca.DATA_DIR

print(f"✅ Agent ready (model: {MODEL}, data dir: {DATA_DIR})")
print("   No files loaded yet — data is loaded on demand when you ask.")

✅ LLM Provider: cborg | Model: claude-sonnet
✅ LLM Provider: cborg | Model: claude-sonnet
✅ Agent ready (model: claude-sonnet, data dir: 731_Data)
   No files loaded yet — data is loaded on demand when you ask.


In [2]:
# ── Tools & Agent (imported from chat_app.py) ────────────────────────────────
# All tool definitions, implementations, dispatch, and system prompt are
# maintained in chat_app.py. We import them here to avoid duplication.

TOOLS = _ca.TOOLS
TOOL_DISPATCH = _ca.TOOL_DISPATCH
SYSTEM_PROMPT = _ca.SYSTEM_PROMPT

conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

tool_names = [t['function']['name'] for t in TOOLS]
print(f"✅ Tools & agent ready.")
print(f"   Available tools: {', '.join(tool_names)}")


def chat(user_message: str):
    """Send a natural-language message to the agent."""
    conversation.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL, messages=conversation, tools=TOOLS, tool_choice="auto",
    )
    msg = response.choices[0].message

    while msg.tool_calls:
        conversation.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"🔧 {fn_name}({json.dumps(fn_args)})")
            fn = TOOL_DISPATCH.get(fn_name)
            try:
                result = fn(**fn_args) if fn else f"Error: Unknown tool '{fn_name}'"
            except Exception as exc:
                result = f"Error in {fn_name}: {exc}"
            conversation.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        response = client.chat.completions.create(
            model=MODEL, messages=conversation, tools=TOOLS, tool_choice="auto",
        )
        msg = response.choices[0].message

    conversation.append(msg)
    if msg.content:
        display(Markdown(msg.content))

✅ Tools & agent ready.
   Available tools: list_scans, plot_scan, compare_scans, save_data, save_image, show_scan_info, normalize_scan, derivative_scan, find_peaks_scan, identify_edge, rename_scan, calibrate_scans, plot_file


---
## 💬 Talk to the Agent

Run the cell below to launch the **interactive chat app** in your browser.

A new tab will open at [http://localhost:5050](http://localhost:5050) with a full chat interface.

Press **Interrupt** (⬛) in the notebook toolbar to stop the server when done.

In [4]:
# ── Launch the Chat Web App ───────────────────────────────────────────────────
import subprocess, webbrowser, time, socket

PORT = 5050

# Start the Flask server as a subprocess
proc = subprocess.Popen(
    ['python3', 'chat_app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

print('🚀 Starting XAS Agent Chat server...')

# Wait until the server is actually listening
for _ in range(30):  # up to 15 seconds
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except (ConnectionRefusedError, OSError):
        time.sleep(0.5)
else:
    print('❌ Server did not start in time. Check chat_app.py for errors.')

print(f'✅ Server running at http://localhost:{PORT}')
print('   Press the ⬛ Stop button to shut down.')
webbrowser.open(f'http://localhost:{PORT}')

# Stream server output until interrupted
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print('\n🛑 Shutting down...')
finally:
    proc.terminate()
    proc.wait()

🚀 Starting XAS Agent Chat server...
✅ Server running at http://localhost:5050
   Press the ⬛ Stop button to shut down.
✅ LLM Provider: cborg | Model: claude-sonnet
  🤖 XAS Agent Chat
  Open http://localhost:5050 in your browser
  Press Ctrl+C to stop
 * Serving Flask app 'chat_app'
 * Debug mode: off
 * Running on http://127.0.0.1:5050
Press CTRL+C to quit
127.0.0.1 - - [08/Apr/2026 23:24:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:24:08] "GET /api/calibration HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:24:08] "GET /api/files HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:25:03] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:27:11] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:27:43] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:27:51] "GET /api/files HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:28:57] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:29:30] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 23:30:52] "POST 